# Pb+Pb 0–10 %: the extra hard scatterings in the hadrons and in unsubtracted jets — `hist` version

The same analysis as `extra_hard_bulk_jets.ipynb` (kept as it is), written with
[`hist`](https://hist.readthedocs.io) and [`mplhep`](https://mplhep.readthedocs.io): the same
input (`RUN/data/extra_hard_hists.npz`, from `extra_hard_hists.py`), the same numbers.
`RUN` is the campaign's git-ignored run folder `../runs/pbpb_extra_hard/`; the campaign, its
cost and the flow analysis are in `PBPB_EXTRA_HARD.md` there.

| | |
|---|---|
| **bulk** | `bulk_bg`, sample k of the event's background: no extras |
| **bulk + wakes** | `bulk_jet`, sample k of the jet leg: the extras' energy deposited in MUSIC_2 |
| **bulk + extras** | `bulk_jet` + `extra_frag`: plus the extras' own (not thermalized) hadrons |

**What `hist` does here, and what it does not:**

* Every observable is one `Hist` with a **`block` axis** (the 100 jackknife blocks of hydro
  backgrounds) and a named physics axis (`eta`, `phi`, `dphi2`, `pt`, `jet_pt`); each block
  holds the sum of its events' per-oversample means. The mean is the projection over `block`
  divided by the number of events, a jackknife replicate is the total minus one block. This is
  exactly the per-event jackknife of the first version (checked below).
* The **errors are the jackknife's**, stored as the variance of a `Weight` histogram (as
  `zjet_mixing.observables_hist` does). hist's own √Σw² would be wrong: an event's 40
  oversamples share one fluid, and bulk and bulk + wakes were sampled with the same random
  numbers, so only replicates of the ratio give its error.
* `hist` handles the axes (names, labels, edges travel with the data), **rebinning**
  (`::hist.rebin(2)` before the jackknife), **selection by value** (`hist.loc`), **profiles**
  (`storage.Mean`), arithmetic between histograms (`a / b`), and mplhep draws them
  (`binwnorm=1` for densities).
* The results are written to `RUN/data/extra_hard_results_hist.pkl` (a dict of `Hist`).

**Contents:** 1. the extras · 2. η and φ of charged hadrons · 3. charged-hadron pT spectra ·
4. anti-kT R = 0.4 jets, unsubtracted · 5. results file · 6. summary

In [ ]:
import glob, json, os, pickle
import numpy as np
import matplotlib.pyplot as plt
import mplhep
import hist
from hist import Hist
from scipy.stats import poisson

RUN = os.path.join("..", "runs", "pbpb_extra_hard")      # the campaign's git-ignored run folder
d = dict(np.load(os.path.join(RUN, "data", "extra_hard_hists.npz")))
N = len(d["seed"])

STYLE = {   # fixed series colours (validated categorical slots 1-4) + line style as second cue
    "bg":    dict(color="#2a78d6", ls="-",  label="bulk"),
    "jet":   dict(color="#eb6834", ls="--", label="bulk + wakes"),
    "full":  dict(color="#1baf7a", ls="-.", label="bulk + extras (wakes + extra_frag)"),
    "extra": dict(color="#eda100", ls=":",  label="extra_frag alone"),
}
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "lines.linewidth": 2, "legend.frameon": False, "font.size": 10})

# the physics axes: name, label, edges (the pass's binning)
AXES = {
    "eta":   hist.axis.Variable(d["edges_eta"], name="eta", label=r"$\eta$"),
    "phi":   hist.axis.Variable(d["edges_phi"], name="phi", label=r"$\varphi$ (lab) [rad]"),
    "dphi2": hist.axis.Variable(d["edges_dphi2"], name="dphi2",
                                label=r"$\varphi - \Psi_2$ (mod $\pi$) [rad]"),
    "pt":    hist.axis.Variable(d["edges_pt"], name="pt", label=r"$p_T$ [GeV]"),
    "jet":   hist.axis.Variable(d["edges_jet"], name="jet_pt",
                                label=r"jet $p_T$ (unsubtracted) [GeV]"),
}

# jackknife blocks: the hydro backgrounds, at most 100 contiguous blocks (as analysis/bulk/flow.py)
u, inv = np.unique(d["group"], return_inverse=True)
blk_of_group = np.zeros(len(u), int)
for i, part in enumerate(np.array_split(np.arange(len(u)), min(100, len(u)))):
    blk_of_group[part] = i
BLOCK = blk_of_group[inv]
NB = BLOCK.max() + 1
N_B = np.bincount(BLOCK, minlength=NB)            # events per block
BLOCK_AXIS = hist.axis.Integer(0, NB, name="block", label="jackknife block",
                               underflow=False, overflow=False)

def block_hist(key, axis):
    # per-event histograms (N, bins) -> Hist(block, axis): the sum of each block's events
    h = Hist(BLOCK_AXIS, AXES[axis], storage=hist.storage.Double())
    v = np.zeros((NB, len(AXES[axis])))
    np.add.at(v, BLOCK, d[key])
    h.view()[...] = v
    return h

KEYS = {"eta": "eta", "phi": "phi", "dphi2": "dphi2", "pt": "pt",
        "jet_ch": "jet", "jet_all": "jet", "lead_ch": "jet", "lead_all": "jet"}
H = {}                                           # H[q][composition]: Hist(block, axis)
for q, ax in KEYS.items():
    H[q] = {c: block_hist(f"{c}.{q}", ax) for c in ("bg", "jet", "full")}
    H[q]["extra"] = H[q]["full"] - H[q]["jet"]

def mean(h):
    return h.project(*[a.name for a in h.axes if a.name != "block"]) / N

def replicate(h, b):
    return (h[{"block": sum}] - h[{"block": b}]) / (N - N_B[b])

def jackknife(f, *hs):
    # f(*mean hists) and its jackknife over the blocks.  f returns a Hist -> a Weight Hist
    # with the jackknife variance; f returns numbers -> (value, error) arrays.
    val = f(*[mean(h) for h in hs])
    reps = [f(*[replicate(h, b) for h in hs]) for b in range(NB)]
    rv = np.array([r.values() if isinstance(r, Hist) else np.asarray(r) for r in reps], float)
    rv[~np.isfinite(rv)] = np.nan                # a block that empties a bin: no error there
    with np.errstate(invalid="ignore"):
        var = (NB - 1) / NB * ((rv - rv.mean(0)) ** 2).sum(0)
    if not isinstance(val, Hist):
        return np.asarray(val), np.sqrt(var)
    out = Hist(*val.axes, storage=hist.storage.Weight())
    out.view().value = val.values()
    out.view().variance = var
    return out

def ratio(a, b):
    with np.errstate(divide="ignore", invalid="ignore"):
        return a / b

print(f"{N} events, {len(u)} backgrounds in {NB} jackknife blocks; e.g.", H["jet_ch"]["bg"])

In [ ]:
# check: the block histograms give the per-event jackknife of the first version exactly
def jk_events(f, *arrays):
    val = f(*[a.mean(0) for a in arrays])
    reps = np.array([f(*[a[BLOCK != b].mean(0) for a in arrays]) for b in range(NB)])
    return val, np.sqrt((NB - 1) / NB * ((reps - reps.mean(0)) ** 2).sum(0))
r_h = jackknife(ratio, H["lead_ch"]["full"], H["lead_ch"]["bg"])
v_e, e_e = jk_events(lambda a, b: np.where(b > 0, a / np.where(b > 0, b, 1), np.nan),
                     d["full.lead_ch"], d["bg.lead_ch"])
ok = np.isfinite(v_e)
print("max |difference| of value, error:",
      np.nanmax(abs(r_h.values()[ok] - v_e[ok])), np.nanmax(abs(np.sqrt(r_h.variances()[ok]) - e_e[ok])))

## 1. The extras

N ~ Poisson(λ) per event, λ = N_coll σ(p̂T > 15 GeV)/σ_NN with each job's warm-up σ; the
expected distribution of N is the Poisson mixture over the events' λ. Events are independent
in their extras (each event draws its own), so here hist's own √Σw² is the right error. The
mean number against N_coll is a profile (`storage.Mean`).

In [ ]:
SIG = {}
for p in glob.glob(os.path.join(RUN, "jobs", "*_seed*.json")):
    j = json.load(open(p)); SIG[j["seed"]] = (j["extra_sigma"], j["extra_sigma_nn"])
lam = np.array([d["n_coll"][i] * SIG[s][0] / SIG[s][1] for i, s in enumerate(d["seed"])])

h_n = Hist(hist.axis.Integer(0, 30, name="n", label="extra hard scatterings per event")).fill(d["n_extra"])
h_prof = Hist(hist.axis.Regular(8, 1100, 2160, name="ncoll", label=r"$N_{coll}$"),
              storage=hist.storage.Mean()).fill(d["n_coll"], sample=d["n_extra"])
h_ni = Hist(hist.axis.Regular(32, 0, 128, name="n", label="initiators of the extras per event")).fill(d["n_initiators"])
w = np.full(len(d["extra_pthat"]), 1 / N)
h_pthat = Hist(hist.axis.Regular(35, 15, 120, name="pthat", label=r"$\hat p_T$ of the extras [GeV]"),
               storage=hist.storage.Weight()).fill(d["extra_pthat"], weight=w)
wi = np.full(len(d["ini_pt"]), 1 / N)
h_ipt = Hist(hist.axis.Variable(np.concatenate([np.arange(0, 20, 1.0), np.arange(20, 60, 4.0),
                                                np.arange(60, 141, 10.0)]),
                                name="pt", label=r"initiator $p_T$ [GeV]"),
             storage=hist.storage.Weight()).fill(d["ini_pt"], weight=wi)
h_ieta = Hist(hist.axis.Regular(40, -8, 8, name="eta", label=r"initiator $\eta$"),
              hist.axis.Variable([0, 10, np.inf], name="pt", label=r"$p_T$"),
              storage=hist.storage.Weight()).fill(d["ini_eta"], d["ini_pt"], weight=wi)

print(f"extras per event: mean {d['n_extra'].mean():.2f}, variance {d['n_extra'].var():.2f} "
      f"(expected {lam.mean():.2f}, {lam.mean() + lam.var():.2f});  initiators per scattering "
      f"{d['n_initiators'].sum() / d['n_extra'].sum():.2f}")

fig, ax = plt.subplots(2, 3, figsize=(13, 7.2))
a = ax[0, 0]
mplhep.histplot(h_n, ax=a, histtype="fill", color=STYLE["bg"]["color"], alpha=0.8, label="campaign",
                flow="none")
k = h_n.axes[0].centers
a.plot(k, poisson.pmf(np.floor(k)[:, None], lam[None, :]).mean(1) * N, "o", color="#2b2b2b", ms=4,
       label="Poisson(λ), mixed over the events")
a.set(xlabel=h_n.axes[0].label, ylabel="events", xlim=(0, 30), ylim=(0, 135))
a.set_xticks([t + 0.5 for t in range(0, 30, 5)], labels=[str(t) for t in range(0, 30, 5)])
a.legend(loc="upper right", fontsize=8)
a = ax[0, 1]
a.scatter(d["n_coll"], d["n_extra"], s=6, alpha=0.3, color=STYLE["bg"]["color"], label="events")
c = h_prof.axes[0].centers
a.errorbar(c, h_prof.values(), np.sqrt(h_prof.variances()), fmt="o-", color="#2b2b2b", ms=5,
           label="profile (storage.Mean)")
a.plot(c, c * np.mean([s[0] / s[1] for s in SIG.values()]), color=STYLE["jet"]["color"], ls="--",
       label=r"$N_{coll}\,\sigma/\sigma_{NN}$")
a.set(xlabel=h_prof.axes[0].label, ylabel="extras per event"); a.legend(loc="upper left", fontsize=8)
a = ax[0, 2]
mplhep.histplot(h_ni, ax=a, histtype="fill", color=STYLE["bg"]["color"], alpha=0.8, flow="none")
a.set(xlabel=h_ni.axes[0].label, ylabel="events")
for a, h, yl in ((ax[1, 0], h_pthat, r"$dN/d\hat p_T$ per event [1/GeV]"),
                 (ax[1, 1], h_ipt, r"$dN/dp_T$ per event [1/GeV]")):
    mplhep.histplot(h, ax=a, histtype="errorbar", binwnorm=1, color=STYLE["bg"]["color"],
                    markersize=4, flow="none")
    a.set(yscale="log", xlabel=h.axes[0].label, ylabel=yl)
a = ax[1, 2]
for sl, st, lab in ((sum, "bg", r"initiators, $p_T$ > 0 GeV"), (1, "jet", r"initiators, $p_T$ > 10 GeV")):
    hh = h_ieta[:, sl] if sl is sum else h_ieta[:, sl:: sum]
    mplhep.histplot(hh, ax=a, binwnorm=1, color=STYLE[st]["color"], ls=STYLE[st]["ls"], label=lab,
                    flow="none")
a.axvspan(-2.4, 2.4, color="#888", alpha=0.12, label=r"hadron acceptance $|\eta|$ < 2.4")
a.set(xlabel=r"initiator $\eta$", ylabel=r"dN/d$\eta$ per event", ylim=(0, 10.5))
a.legend(loc="upper right", fontsize=7)
for x, l in zip(ax.flat, "abcdef"):
    x.set_title(f"({l})", loc="left", fontsize=10)
fig.tight_layout()

## 2. η and φ of charged hadrons

Charged hadrons with pT > 0.15 GeV (η over |η| < 2.6, φ in |η| < 2.4). The lab φ carries the
event's flow (the impact parameter has a fixed direction); φ − Ψ₂ is against the background's
Ψ₂ from all its oversamples, the same Ψ₂ for the three compositions.

In [ ]:
RESULTS = {}

def plot_with_ratios(q, comps=("bg", "jet", "full"), ratios=(("jet", "bg"), ("full", "bg")),
                     rebin=None, ylabel="", logy=False, rlog=False, rlim=None, xlim=None,
                     title=None, xlog=False):
    hs = {c: H[q][c] for c in comps}
    if rebin:
        hs = {c: h[:, ::hist.rebin(rebin)] for c, h in hs.items()}
    fig, (a0, a1) = plt.subplots(2, 1, figsize=(7, 6.2), sharex=True,
                                 gridspec_kw=dict(height_ratios=[2, 1.3], hspace=0.06))
    for c in comps:
        m = jackknife(lambda h: h, hs[c])
        RESULTS[f"{q}/{c}"] = m
        mplhep.histplot(m, ax=a0, binwnorm=1, yerr=np.sqrt(m.variances()), edges=False,
                        flow="none", **STYLE[c])
    for num, den in ratios:
        r = jackknife(ratio, hs[num], hs[den])
        RESULTS[f"{q}/{num}_over_{den}"] = r
        mplhep.histplot(r, ax=a1, histtype="errorbar", yerr=np.sqrt(r.variances()), flow="none",
                        color=STYLE[num]["color"], markersize=3.5,
                        label=f"{STYLE[num]['label']} / {STYLE[den]['label']}")
    a1.axhline(1, color="#555", lw=1)
    a0.set(ylabel=ylabel, yscale="log" if logy else "linear", xlabel="")
    a1.set(ylabel="ratio", xlabel=hs[comps[0]].axes[1].label,
           yscale="log" if rlog else "linear")
    for a in (a0, a1):
        if xlog:
            a.set_xscale("log")
    if rlim: a1.set_ylim(*rlim)
    if xlim: a1.set_xlim(*xlim)
    if title: a0.set_title(title, fontsize=10)
    a0.legend(fontsize=8); a1.legend(fontsize=8)
    return fig

plot_with_ratios("eta", ylabel=r"dN/d$\eta$ (charged, $p_T$ > 0.15 GeV)",
                 title="charged hadrons, all events");

In [ ]:
plot_with_ratios("phi", ylabel=r"dN/d$\varphi$ (charged, $|\eta|$ < 2.4)");
plot_with_ratios("dphi2", ylabel=r"dN/d$\varphi$ (charged, $|\eta|$ < 2.4)");

# the ratio's modulation R = a (1 + 2 b cos 2Δφ): b ≈ Δv₂ relative to Ψ₂
c = AXES["dphi2"].centers
def fit_mod(num, den):
    r = (num / den).values()
    a, b2 = np.linalg.lstsq(np.vstack([np.ones_like(c), np.cos(2 * c)]).T, r, rcond=None)[0]
    return np.array([a, b2 / (2 * a)])
v2bg, ev2 = jackknife(lambda h: (h.values() * np.cos(2 * c)).sum() / h.sum(), H["dphi2"]["bg"])
for nm in ("jet", "full"):
    (a, b), (ea, eb) = jackknife(fit_mod, H["dphi2"][nm], H["dphi2"]["bg"])
    print(f"{STYLE[nm]['label']:>35s} / bulk:  a = {a:.4f} ± {ea:.4f},  b = {b:+.5f} ± {eb:.5f}"
          f"  -> Δv₂/v₂ ≈ {b / v2bg:+.3f} ± {eb / v2bg:.3f}  (bulk <cos 2(φ−Ψ₂)> = {v2bg:.4f})")

## 3. Charged-hadron pT spectra, |η| < 0.8

In [ ]:
plot_with_ratios("pt", comps=("bg", "jet", "full", "extra"), logy=True, rlog=True, rlim=(0.9, 200),
                 xlog=True, ylabel=r"dN/d$p_T$ per event [1/GeV]");
f_ex = jackknife(ratio, H["pt"]["extra"], H["pt"]["full"])
RESULTS["pt/extra_fraction"] = f_ex
print("fraction of extra_frag in the full event's charged hadrons:")
for x in (1, 2, 4, 6, 10, 20):
    i = f_ex.axes[0].index(x)
    print(f"   pT in [{f_ex.axes[0].edges[i]:.2f}, {f_ex.axes[0].edges[i + 1]:.2f}) GeV: "
          f"{f_ex.values()[i]:.3f} ± {np.sqrt(f_ex.variances()[i]):.3f}")
half = f_ex.axes[0].centers[np.argmax(f_ex.values() > 0.5)]
print(f"extra_frag is more than half above pT ≈ {half:.1f} GeV")

## 4. Jets: anti-kT R = 0.4, unsubtracted

FastJet anti-kT R = 0.4 on each oversample, **no background subtraction**: constituents
pT > 0.15 GeV, |η| < 2.4; jets |η| < 2.0; charged-particle jets and jets of all hadrons
(π⁰ and η are stable in our decays). Here the spectra are **rebinned ×2 (12 GeV bins) before
the jackknife** (`::hist.rebin(2)`): the tails are short of statistics. The tail probabilities
below use the original 6 GeV bins and select by value with `hist.loc`.

In [ ]:
XMAX = {"ch": 270, "all": 410}
for kind, nm in (("ch", "charged-particle jets"), ("all", "jets of all hadrons")):
    plot_with_ratios(f"jet_{kind}", rebin=2, logy=True, rlog=True, rlim=(0.5, 20), xlim=(0, XMAX[kind]),
                     ylabel=r"dN/d$p_T$ per event [1/GeV]",
                     title=f"inclusive {nm}, R = 0.4, |η_jet| < 2.0, 12 GeV bins")

In [ ]:
for kind, nm in (("ch", "charged-particle jets"), ("all", "jets of all hadrons")):
    plot_with_ratios(f"lead_{kind}", rebin=2, logy=True, rlog=True, rlim=(0.3, 30), xlim=(50, XMAX[kind]),
                     ylabel="probability density [1/GeV]",
                     title=f"leading {nm} per oversample, R = 0.4, 12 GeV bins")

In [ ]:
# tails: P(leading jet of an oversample above x), x by value (the bin that contains x starts there)
def above(x):
    return lambda h: h[hist.loc(x)::sum]
def above_ratio(x):
    return lambda a, b: a[hist.loc(x)::sum] / b[hist.loc(x)::sum]
print(f"{'jets':>4s} {'> pT':>5s} {'P(bulk)':>18s} {'P(bulk + extras)':>20s} {'wakes / bulk':>14s} {'extras / bulk':>15s}")
for kind, ths in (("ch", (120, 150, 168, 186)), ("all", (240, 270, 300, 318))):
    for x in ths:
        lo = AXES["jet"].edges[AXES["jet"].index(x)]
        pb, epb = jackknife(above(x), H[f"lead_{kind}"]["bg"])
        pf, epf = jackknife(above(x), H[f"lead_{kind}"]["full"])
        rj, erj = jackknife(above_ratio(x), H[f"lead_{kind}"]["jet"], H[f"lead_{kind}"]["bg"])
        rf, erf = jackknife(above_ratio(x), H[f"lead_{kind}"]["full"], H[f"lead_{kind}"]["bg"])
        print(f"{kind:>4s} {lo:5.0f} {pb:9.2e} ± {epb:7.1e} {pf:10.2e} ± {epf:7.1e} "
              f"{rj:6.2f} ± {erj:4.2f} {rf:7.2f} ± {erf:4.2f}")

## 5. The results as `Hist` objects

Every plotted mean and ratio, with its jackknife variance (`storage.Weight`), in one pickle:
`RESULTS["<observable>/<composition>"]` and `RESULTS["<observable>/<num>_over_<den>"]`
(observables `eta`, `phi`, `dphi2`, `pt`, `jet_ch`, `jet_all`, `lead_ch`, `lead_all`;
compositions `bg`, `jet`, `full`, `extra`). The values are per event and per bin, not per GeV
(`binwnorm=1` in mplhep, or divide by `h.axes[0].widths`).

In [ ]:
RESULTS.update({"extras/n": h_n, "extras/n_vs_ncoll": h_prof, "extras/n_initiators": h_ni,
                "extras/pthat": h_pthat, "extras/initiator_pt": h_ipt, "extras/initiator_eta_pt": h_ieta})
path = os.path.join(RUN, "data", "extra_hard_results_hist.pkl")
with open(path, "wb") as f:
    pickle.dump(RESULTS, f)
print(f"wrote {path}: {len(RESULTS)} histograms:", ", ".join(RESULTS))
h = RESULTS["lead_ch/full_over_bg"]
i = h.axes[0].index(186); lo, hi = h.axes[0].edges[i:i + 2]
print(f"e.g. lead_ch/full_over_bg in [{lo:.0f}, {hi:.0f}) GeV (12 GeV bins): "
      f"{h[i].value:.2f} ± {np.sqrt(h[i].variance):.2f}")

## 6. Summary

The results and their discussion are kept with the campaign, in the git-ignored run folder
`../runs/pbpb_extra_hard/`: the executed notebook there and `PBPB_EXTRA_HARD.md`.